# Gemini 스트리밍 응답

Gemini가 생성하는 텍스트를 실시간으로 출력하고, 전달된 조각을 하나의 응답으로 합칩니다.

In [1]:
import os

from dotenv import load_dotenv
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
client = genai.Client(api_key=api_key)
print("준비 완료 / 사용 모델:", model)

준비 완료 / 사용 모델: gemini-3.6-flash


## Generator로 스트리밍 이해하기

일반 HTTP 응답은 서버가 결과를 모두 만든 뒤 한 번에 전달합니다. 스트리밍 응답은 준비된 결과를 여러 조각으로 나누어 바로 전달합니다.

Python Generator는 `yield`를 이용해 값을 하나씩 반환합니다. 아래 코드는 `time.sleep()`으로 응답 조각이 늦게 도착하는 상황을 흉내 냅니다.

In [2]:
import time


def local_stream():
    chunks = ["안녕", "하세요", "! ", "스트리밍", " 응답입니다."]

    for chunk in chunks:
        time.sleep(0.5)  # 데이터가 늦게 도착하는 상황을 흉내 냅니다.
        yield chunk


for chunk in local_stream():
    print(chunk, end="", flush=True)

안녕하세요! 스트리밍 응답입니다.

## SSE(Server-Sent Events)란?

SSE(Server-Sent Events)는 하나의 HTTP 연결을 유지하면서 서버가 클라이언트에 데이터를 계속 전달하는 방식입니다.

Gemini에서 `stream=True`를 지정하면 SSE로 응답을 받습니다. `google-genai` SDK는 전달받은 데이터를 Python 이벤트 객체로 변환해 줍니다.

## 스트리밍 이벤트 확인하기

`stream=True`로 요청하면 완성된 응답 대신 여러 이벤트가 순서대로 전달됩니다. 아래 코드에서는 `for`문으로 이벤트를 하나씩 받아 이벤트 타입을 출력합니다.

In [ ]:
event_stream = client.interactions.create(
    model=model,
    input="봄을 표현하는 짧은 문장을 하나 작성해 줘.",
    stream=True,
    store=False,
)

for event in event_stream:
    
    print(event.event_type)

interaction.created
interaction.status_update
step.start
step.delta
step.stop
step.start
step.delta
step.stop
interaction.completed


## 텍스트 실시간으로 출력하기

응답 내용은 `step.delta` 이벤트를 통해 조각 단위로 전달됩니다. 텍스트 조각만 골라 바로 출력하고 `text_parts`에 저장합니다.

In [4]:
prompt = "AI Agent가 LLM을 여러 번 호출하는 이유를 세 문장으로 설명해 줘."

stream = client.interactions.create(
    model=model,
    input=prompt,
    stream=True,
    store=False,
)

text_parts = []

for event in stream:
    if event.event_type == "step.delta":
        delta = getattr(event, "delta", None)
        if getattr(delta, "type", None) == "text":
            text = getattr(delta, "text", "")
            if text:
                text_parts.append(text)
                print(text, end="", flush=True)

AI 에이전트는 복잡한 목표를 해결하기 위해 전체 과업을 여러 단계로 나누고 순차적으로 실행 계획을 수립합니다. 각 단계에서 외부 도구를 사용한 후, 그 결과나 새로운 정보를 다시 분석하여 다음 행동을 결정해야 합니다. 또한, 최종 결과물이 완성될 때까지 답변의 오류를 스스로 검토하고 수정하는 자가 피드백 과정을 반복하기 때문입니다.

In [5]:
full_text = "".join(text_parts)

print("합쳐진 전체 응답:")
print(full_text)

합쳐진 전체 응답:
AI 에이전트는 복잡한 목표를 해결하기 위해 전체 과업을 여러 단계로 나누고 순차적으로 실행 계획을 수립합니다. 각 단계에서 외부 도구를 사용한 후, 그 결과나 새로운 정보를 다시 분석하여 다음 행동을 결정해야 합니다. 또한, 최종 결과물이 완성될 때까지 답변의 오류를 스스로 검토하고 수정하는 자가 피드백 과정을 반복하기 때문입니다.
